# Val Set Distance Distribution Analysis

학습된 best GRU 모델로 Validation Set에 대해 추론하고,
예측 위치와 실제 라벨 사이의 거리(dist) 분포를 시각화합니다.

In [ ]:
import sys
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

sys.path.insert(0, str(Path('.').resolve()))
from train_gru import MosquitoDataset, MosquitoGRU, Config

In [ ]:
# ===== 설정 =====
# 학습 시 사용한 옵션과 동일하게 맞춰야 합니다
USE_DELTA    = False   # --input delta 로 학습했으면 True
USE_ROTATION = True    # --no-rotate 로 학습했으면 False
SEED         = 42
ACC_THRESHOLD = 0.01   # 정답 인정 거리 기준 (m)

In [ ]:
# Best 모델 자동 선택 (dist 기준 오름차순 → 가장 낮은 것)
def get_dist(path):
    try:
        return float(path.stem.split('_')[1])
    except:
        return float('inf')

model_files = sorted(Path('model').glob('gru_*_*.pth'), key=get_dist)
if not model_files:
    # 구 형식 (dist만 있는 경우) 도 탐색
    model_files = sorted(Path('model').glob('gru_*.pth'), key=lambda p: float(p.stem.split('_')[1]))

MODEL_PATH = model_files[0]
print(f"사용 모델: {MODEL_PATH}")

model = MosquitoGRU(
    input_size=Config.input_size,
    hidden_size=Config.hidden_size,
    num_layers=Config.num_layers,
    output_size=Config.output_size,
)
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'))
model.eval()
print("모델 로드 완료")

In [ ]:
# 학습과 동일한 split으로 val 파일 추출
all_train_files  = sorted(Path('data/train').glob('TRAIN_*.csv'))
train_labels_df  = pd.read_csv('data/train_labels.csv')
_, val_files = train_test_split(all_train_files, test_size=0.2, random_state=SEED)

Config.use_delta    = USE_DELTA
Config.use_rotation = USE_ROTATION
val_dataset = MosquitoDataset(
    val_files, train_labels_df, is_train=True,
    use_delta=USE_DELTA, use_rotation=USE_ROTATION
)
print(f"Val 샘플 수: {len(val_dataset)}")

In [ ]:
# 전체 val set 추론 → 원본 좌표계로 역회전
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

all_preds, all_targets = [], []
with torch.no_grad():
    for seq, target in val_loader:
        all_preds.append(model(seq).numpy())
        all_targets.append(target.numpy())

preds_rot   = np.concatenate(all_preds,   axis=0)  # (N, 3)
targets_rot = np.concatenate(all_targets, axis=0)  # (N, 3)

last_pos = np.array(val_dataset.last_positions)  # (N, 3)
rot_mats = np.array(val_dataset.rot_mats)         # (N, 3, 3)

# pred_rot @ R  →  original space displacement
pred_pos = last_pos + np.einsum('ni,nij->nj', preds_rot,   rot_mats)
true_pos = last_pos + np.einsum('ni,nij->nj', targets_rot, rot_mats)

dists = np.linalg.norm(pred_pos - true_pos, axis=1)  # (N,)

n_correct = (dists < ACC_THRESHOLD).sum()
print(f"Val Dist  mean : {dists.mean():.4f} m")
print(f"Val Dist median: {np.median(dists):.4f} m")
print(f"Val Acc (< {ACC_THRESHOLD} m): {n_correct}/{len(dists)}  ({n_correct/len(dists)*100:.2f}%)")

In [ ]:
# 거리 분포 히스토그램
BIN_SIZE = 0.001  # 히스토그램 bin 너비 (m)

fig = go.Figure()

# 정답(< threshold) / 오답(>= threshold) 색상 구분
correct_dists = dists[dists <  ACC_THRESHOLD]
wrong_dists   = dists[dists >= ACC_THRESHOLD]

fig.add_trace(go.Histogram(
    x=correct_dists, name=f'Correct (< {ACC_THRESHOLD} m)',
    xbins=dict(size=BIN_SIZE), marker_color='steelblue', opacity=0.85,
))
fig.add_trace(go.Histogram(
    x=wrong_dists, name=f'Wrong (≥ {ACC_THRESHOLD} m)',
    xbins=dict(size=BIN_SIZE), marker_color='tomato', opacity=0.85,
))

# threshold / mean / median 수직선
fig.add_vline(x=ACC_THRESHOLD, line_dash='dash', line_color='red', line_width=2,
              annotation_text=f'threshold {ACC_THRESHOLD} m',
              annotation_position='top right', annotation_font_size=12)
fig.add_vline(x=dists.mean(), line_dash='dot', line_color='orange', line_width=1.5,
              annotation_text=f'mean {dists.mean():.4f} m',
              annotation_position='top left', annotation_font_size=11)
fig.add_vline(x=np.median(dists), line_dash='dot', line_color='green', line_width=1.5,
              annotation_text=f'median {np.median(dists):.4f} m',
              annotation_position='bottom right', annotation_font_size=11)

fig.update_layout(
    title=(
        f'Val Set Distance Distribution  |  '
        f'Acc = {n_correct}/{len(dists)} ({n_correct/len(dists)*100:.2f}%)'
    ),
    xaxis_title='Euclidean Distance Error (m)',
    yaxis_title='Count',
    barmode='overlay',
    legend=dict(yanchor='top', y=0.99, xanchor='right', x=0.99),
    width=850, height=500,
)
fig.show()

In [ ]:
# CDF (누적 분포) — 특정 거리 이하에 몇 %가 들어오는지 한눈에 확인
sorted_dists = np.sort(dists)
cdf = np.arange(1, len(sorted_dists) + 1) / len(sorted_dists)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=sorted_dists, y=cdf * 100,
    mode='lines', name='CDF', line=dict(color='steelblue', width=2),
))
fig2.add_vline(x=ACC_THRESHOLD, line_dash='dash', line_color='red', line_width=2,
               annotation_text=f'{ACC_THRESHOLD} m → {n_correct/len(dists)*100:.1f}%',
               annotation_position='top left', annotation_font_size=12)

fig2.update_layout(
    title='Cumulative Distribution of Val Distance Error',
    xaxis_title='Distance Error (m)',
    yaxis_title='Cumulative % of samples',
    width=850, height=450,
)
fig2.show()